In [ ]:
from pandas.core.frame import DataFrame
import pandas as pd
import os

symbol = "VNM"
daily_file = f"processed_stck_data/{symbol}_RRV.csv"
df_daily: DataFrame = pd.read_csv(daily_file, parse_dates=["Date"])

In [ ]:
from vnstock3 import Vnstockm 
stock = Vnstock().stock(symbol=symbol, source='TCBS')

df_quote_history= stock.quote.history(start='2025-01-01', end='2025-01-31')
df_cash_flow = stock.finance.cash_flow(period='quarter') 
df_income_statement = stock.finance.income_statement(period='quarter')
df_ratio = stock.finance.ratio(period='quarter')

df_quote_history.to_csv(f'files/{symbol}_2025_quote_history.csv', index=False)
df_cash_flow.to_csv(f'files/{symbol}_2025_cash_flow.csv', index=False)
df_income_statement.to_csv(f'files/{symbol}_2025_income_statement.csv', index=False)
df_ratio.to_csv(f'files/{symbol}_2025_ratio.csv', index=False)
print('done!')

In [ ]:
start_date = pd.to_datetime("2013-03-31")
end_date   = pd.to_datetime("2024-12-31")

In [ ]:
# Filter daily data to this date range
df_daily = df_daily[(df_daily["Date"] >= start_date) & (df_daily["Date"] <= end_date)]
df_daily.sort_values("Date", inplace=True)

In [ ]:
df_daily.head()

In [ ]:
df_daily.tail()

In [ ]:
df_daily.isna().sum()[df_daily.isna().sum() > 0]

In [ ]:
# Mapping for quarter end dates
quarter_end_map = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}

def load_and_process_quarterly(file_path, start_date, end_date):
    df = pd.read_csv(file_path)

    if not {"year", "quarter"}.issubset(df.columns):
        raise ValueError(f"File {file_path} must contain 'year' and 'quarter' columns.")

    df["Quarter_End"] = pd.to_datetime(
        df["year"].astype(str) + df["quarter"].astype(int).map(quarter_end_map)
    )
    
    df = df[(df["Quarter_End"] >= start_date) & (df["Quarter_End"] <= end_date)]
    
    df.sort_values("Quarter_End", inplace=True)
    return df

In [ ]:
cash_flow_file = f"files/{symbol}_2025_cash_flow.csv"
financial_reports_file = f"files/{symbol}_2025_income_statement.csv"
stock_ratio_file = f"files/{symbol}_2025_ratio.csv"

start_date = pd.to_datetime("2013-03-31")
end_date   = pd.to_datetime("2024-12-31")

df_cash_flow = load_and_process_quarterly(cash_flow_file, start_date, end_date)
df_financial  = load_and_process_quarterly(financial_reports_file, start_date, end_date)
df_stock_ratio = load_and_process_quarterly(stock_ratio_file, start_date, end_date)

In [ ]:
df_merged = pd.merge_asof(
    df_daily.sort_values("Date"),
    df_cash_flow,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_cf")
)

df_merged = pd.merge_asof(
    df_merged.sort_values("Date"),
    df_financial,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_fr")
)

df_merged = pd.merge_asof(
    df_merged.sort_values("Date"),
    df_stock_ratio,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_sr")
)

In [ ]:
len(df_merged.columns.tolist())

In [ ]:
df_merged.columns.tolist()

In [ ]:
cols_to_drop = ['Quarter_End_sr', 'year_sr', 'quarter_sr', 'Quarter_End_fr', 'year_fr', 'quarter_fr', 'Quarter_End', 'year', 'quarter' ]
df_merged.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# Step 1: Create a mapping of old → new names
original_columns = [
    'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change', 'Symbol',
    'return_day', 'return_week', 'return_month', 'volatility_day', 'volatility_week', 'volatility_month',
    'liquidity_day', 'liquidity_week', 'liquidity_month', 'high_minus_close', 'low_minus_open',
    'cumulative_return', 'Stochastic_Osc', 'ATR', 'SMA_3', 'SMA_7', 'SMA_14', 'SMA_21', 'SMA_50', 'SMA_100',
    'WMA_3', 'WMA_7', 'WMA_14', 'WMA_21', 'WMA_50', 'WMA_100', 'EMA6', 'EMA12', 'EMA26',
    'outMACD', 'outMACDSignal', 'outMACDHist', 'RSI6', 'RSI12', 'RSI14',
    'StochRSI_6', 'StochRSI_12', 'StochRSI_14',
    'BBANDSMIDDLE', 'BBANDSUPPER', 'BBANDSLOWER',
    'OBV', 'MFI14', 'MOM1', 'MOM3', 'MOM7', 'CCI12', 'CCI20', 'ROCR3', 'ROCR12', 'WILLR', 'TRIX'
]

renamed_columns = [
    'date', 'close', 'open', 'high', 'low', 'volume', 'change', 'symbol',
    'return_day', 'return_week', 'return_month', 'volatility_day', 'volatility_week', 'volatility_month',
    'liquidity_day', 'liquidity_week', 'liquidity_month', 'high_minus_close', 'low_minus_open',
    'cumulative_return', 'stochastic_osc', 'atr', 'sma_3', 'sma_7', 'sma_14', 'sma_21', 'sma_50', 'sma_100',
    'wma_3', 'wma_7', 'wma_14', 'wma_21', 'wma_50', 'wma_100', 'ema_6', 'ema_12', 'ema_26',
    'out_macd', 'out_macd_signal', 'out_macd_hist', 'rsi_6', 'rsi_12', 'rsi_14',
    'stochrsi_6', 'stochrsi_12', 'stochrsi_14',
    'bbands_middle', 'bbands_upper', 'bbands_lower',
    'obv', 'mfi_14', 'mom_1', 'mom_3', 'mom_7', 'cci_12', 'cci_20', 'rocr_3', 'rocr_12', 'willr', 'trix'
]

rename_map = dict(zip(original_columns, renamed_columns))
df_merged.rename(columns=rename_map, inplace=True)

In [ ]:
# ======== Step 5: Save the combined DataFrame ========
output_file = f"feature_engineered/feature_engineered_{symbol}_RRV.csv"
df_merged.to_csv(output_file, index=False)
print(f"Saved combined feature-engineered file to {output_file}")

# The End